# Test 1 - Captura direta com OpenCV/GStreamer

Este teste abre a camara CSI atraves de uma pipeline GStreamer usada diretamente pelo OpenCV, captura apenas uma imagem e mostra-a no notebook. Destaca-se por ser a abordagem mais baixa-nivel: permite validar se a pipeline da camara funciona sem depender da biblioteca JetCam.

In [ ]:
from PIL import Image
from IPython.display import display
import cv2
import time


gst_pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), width=1280, height=720, "
    "format=NV12, framerate=60/1 ! "
    "nvvidconv ! "
    "video/x-raw, width=640, height=360, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "appsink drop=true max-buffers=1"
)


cap = cv2.VideoCapture(gst_pipeline, cv2.CAP_GSTREAMER)
print("Camera aberta:", cap.isOpened())

ret, frame = cap.read()
cap.release()

if ret:
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    display(Image.fromarray(frame))

# Test 2 - Captura simples com JetCam

Este teste usa a classe `CSICamera` da biblioteca JetCam para capturar uma unica frame e converte-la para JPEG antes de a mostrar. Destaca-se do Teste 1 por esconder a configuracao GStreamer e oferecer uma interface mais simples, adequada para exemplos rapidos com o JetRacer.

In [ ]:
from IPython.display import display, Image
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

# 1. Inicializar a câmara (se já não estiver inicializada)
camera = CSICamera(width=224, height=224, capture_width=1280, capture_height=720, capture_fps=30)

# 2. Capturar uma única frame
frame = camera.read()

# 3. Converter a frame para bytes JPEG
jpeg_bytes = bgr8_to_jpeg(frame)

# 4. Mostrar a imagem estática no Jupyter Notebook
display(Image(data=jpeg_bytes))


camera.cap.release() # Se expuser o objeto VideoCapture interno
# Ou simplesmente reinicie o kernel se estiver a usar Jupyter Notebooks


# Test 3 - Video no notebook com OpenCV e widget

Este teste usa novamente OpenCV/GStreamer, mas atualiza continuamente um `widgets.Image` com varias frames codificadas em JPEG. Destaca-se por testar visualizacao quase em tempo real mantendo controlo direto sobre a pipeline da camara.

In [ ]:
import cv2
import ipywidgets as widgets
from IPython.display import display
import time

gst_pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), width=1280, height=720, "
    "format=NV12, framerate=60/1 ! "
    "nvvidconv ! "
    "video/x-raw, width=640, height=360, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "appsink drop=true max-buffers=1"
)

image_widget = widgets.Image(format='jpeg')
display(image_widget)

cap = cv2.VideoCapture(gst_pipeline, cv2.CAP_GSTREAMER)
print("Camera aberta:", cap.isOpened())

time.sleep(2)

ret = False
frame = None


for i in range(1000):
    ret, frame = cap.read()
    if not ret:
        continue

    _, jpeg = cv2.imencode('.jpg', frame)
    image_widget.value = jpeg.tobytes()
    #print(i)

cap.release()

# Test 4 - Streaming continuo com JetCam

Este teste combina `CSICamera` com um `display_handle` persistente para atualizar a imagem continuamente no notebook ate o utilizador interromper a execucao. Destaca-se por ser a versao de streaming mais simples ao nivel da API, usando JetCam em vez de gerir manualmente a pipeline GStreamer.

In [ ]:
import cv2
import time
from IPython.display import display, Image, clear_output
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

# 1. Inicializar a câmara
camera = CSICamera(width=224, height=224, capture_width=1280, capture_height=720, capture_fps=30)

# 2. Criar um handle de exibição para evitar cintilação (flicker)
display_handle = display(None, display_id=True)

try:
    while True:
        # Capturar a frame atual
        frame = camera.read()
        
        if frame is not None:
            # Converter a frame BGR8 diretamente para os bytes JPEG através do JetCam
            jpeg_bytes = bgr8_to_jpeg(frame)
            
            # Atualizar o display inline de forma persistente
            display_handle.update(Image(data=jpeg_bytes))
            
        # Controlar a taxa de atualização (aprox. 30 FPS)
        time.sleep(0.03)

except KeyboardInterrupt:
    # Captura o STOP do utilizador (Botão de Interromper o Kernel)
    print("Streaming interrompido pelo utilizador.")
    camera.cap.release() # Se expuser o objeto VideoCapture interno
    # Ou simplesmente reinicie o kernel se estiver a usar Jupyter Notebooks


